## 1. Carga de datasets

Se cargan los datasets trabajados en la primera entrega y otros conjuntos adicionales del repositorio suministrado por la asignatura. La integración de estos datos fortalece el análisis porque permite relacionar cobertura y permanencia escolar con conectividad institucional, resultados de Saber 11 y vulnerabilidad social.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator, ClusteringEvaluator

spark = SparkSession.builder.appName("SegundaEntregaProyectoBigData").getOrCreate()

In [0]:
# Carga de archivos principales

dfInternetOficiales = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/1L__Instituciones_Oficiales_Internet.csv",
    header=True, inferSchema=True
)

dfInternetNoOficiales = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/2L__Instituciones_No_Oficiales_Internet.csv",
    header=True, inferSchema=True
)

dfMatricula = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/3L__Matrícula_Por_Municipio.csv",
    header=True, inferSchema=True
)

dfEstadisticas = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/4L__Estadística_Básica_Media.csv",
    header=True, inferSchema=True
)

dfUNIDOS = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/5L__Estrategia_UNIDOS.csv",
    header=True, inferSchema=True
)

dfInternet = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/6L__Internet_Fijo_Tecnología_Segmento.csv",
    header=True, inferSchema=True
)

dfSaber = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/7L__Resultados_Únicos_Saber_11.csv",
    header=True, inferSchema=True
)

print("Filas por dataset:")
for nombre, df_ in [
    ("Internet oficiales", dfInternetOficiales),
    ("Internet no oficiales", dfInternetNoOficiales),
    ("Matrícula", dfMatricula),
    ("Estadísticas", dfEstadisticas),
    ("UNIDOS", dfUNIDOS),
    ("Internet fijo", dfInternet),
    ("Saber 11", dfSaber),
]:
    print(nombre, df_.count(), len(df_.columns))

## 2. Estandarización territorial

Antes de realizar los cruces se unifica el nombre del municipio en una columna estándar llamada `municipio_std`. Esta transformación era necesaria porque los datasets provienen de fuentes distintas y presentan diferencias de tildes, mayúsculas, espacios y nombres de columnas.

In [0]:
def estandarizar_municipio(df, colname, nueva_col="municipio_std"):
    return (df
            .withColumn(nueva_col, F.upper(F.trim(F.col(colname))))
            .withColumn(nueva_col, F.translate(F.col(nueva_col), "ÁÉÍÓÚÄËÏÖÜÑ", "AEIOUAEIOUN")))

# Estandarización por dataset

dfInternetOficiales = estandarizar_municipio(dfInternetOficiales, "municipio")
dfInternetNoOficiales = estandarizar_municipio(dfInternetNoOficiales, "municipio")
dfMatricula = estandarizar_municipio(dfMatricula, "municipio")
dfEstadisticas = estandarizar_municipio(dfEstadisticas, "municipio")
dfUNIDOS = estandarizar_municipio(dfUNIDOS, "NombreMunicipioAtencion")
dfInternet = estandarizar_municipio(dfInternet, "MUNICIPIO")
dfSaber = estandarizar_municipio(dfSaber, "COLE_MCPIO_UBICACION")

for nombre, df_ in [("Matricula", dfMatricula), ("Estadisticas", dfEstadisticas), ("Saber", dfSaber)]:
    print(f"Municipios distintos en {nombre}:", df_.select("municipio_std").distinct().count())

## 3. Filtros y transformaciones finales

En cumplimiento del enunciado se aplican al menos dos filtros y tres transformaciones con su respectiva justificación. La lógica general fue conservar registros comparables, convertir variables críticas a formato numérico y construir indicadores municipales agregados para análisis y modelado.

### 3.1 Filtros aplicados

**Filtro 1.** Se eliminan en Saber 11 los registros cuyo municipio aparece como `BOYACA`, ya que corresponden a una referencia departamental y no municipal, lo cual impide un join territorial correcto.

**Filtro 2.** Se conserva el año más reciente comparable por dataset para construir una fotografía analítica actual. En Estadística Básica se prioriza 2024, en Matrícula se usa el año más reciente disponible y en los demás datasets se agrega al nivel municipal sin requerir una serie completa.

**Filtro 3.** Se eliminan observaciones con métricas imposibles o atípicas, por ejemplo coberturas fuera de rango o velocidades negativas cuando existan por problemas de digitación o casteo.

In [0]:
# Filtro 1: eliminar referencia departamental en Saber

dfSaber = dfSaber.filter(F.upper(F.trim(F.col("municipio_std"))) != "BOYACA")

# Detección de año más reciente disponible
ultimo_anio_est = dfEstadisticas.select(F.max("año")).collect()[0][0] if "año" in [c.lower() for c in dfEstadisticas.columns] else None
ultimo_anio_mat = dfMatricula.select(F.max("AÑO")).collect()[0][0] if "AÑO" in dfMatricula.columns else None
print("Último año matrícula:", ultimo_anio_mat)
print("Último año estadísticas (si existe nombre estándar):", ultimo_anio_est)

In [0]:
# Utilidades para columnas con porcentajes o separadores decimales

def to_double_clean(c):
    return F.regexp_replace(
        F.regexp_replace(F.regexp_replace(F.col(c).cast("string"), "%", ""), "\.", ""), ",", "."
    ).cast("double")

# Columnas clave de Estadísticas
columnas_estadisticas = {
    "COBERTURA_NETA": "cobertura_neta",
    "COBERTURA_BRUTA": "cobertura_bruta",
    "DESERCIÓN": "desercion",
    "DESERCIÓN_MEDIA": "desercion_media",
    "APROBACIÓN": "aprobacion",
    "REPROBACIÓN": "reprobacion",
    "REPITENCIA": "repitencia",
    "COBERTURA_NETA_PRIMARIA": "cobertura_neta_primaria",
    "COBERTURA_NETA_MEDIA": "cobertura_neta_media",
    "TASA_MATRICULACIÓN_5_16": "tasa_matriculacion_5_16",
    "TAMAÑO_PROMEDIO_DE_GRUPO": "tamano_promedio_grupo"
}

for original, nuevo in columnas_estadisticas.items():
    if original in dfEstadisticas.columns:
        dfEstadisticas = dfEstadisticas.withColumn(nuevo, to_double_clean(original))

# Años más recientes y filtros de calidad
if "AÑO" in dfEstadisticas.columns:
    ultimo_anio_est_real = dfEstadisticas.agg(F.max("AÑO")).collect()[0][0]
    dfEstadisticas = dfEstadisticas.filter(F.col("AÑO") == ultimo_anio_est_real)

if "AÑO" in dfMatricula.columns:
    ultimo_anio_mat_real = dfMatricula.agg(F.max("AÑO")).collect()[0][0]
    dfMatricula = dfMatricula.filter(F.col("AÑO") == ultimo_anio_mat_real)

# Filtro 3: rangos plausibles en estadísticas
if "cobertura_bruta" in dfEstadisticas.columns:
    dfEstadisticas = dfEstadisticas.filter((F.col("cobertura_bruta").isNull()) | ((F.col("cobertura_bruta") >= 0) & (F.col("cobertura_bruta") <= 250)))
if "cobertura_neta" in dfEstadisticas.columns:
    dfEstadisticas = dfEstadisticas.filter((F.col("cobertura_neta").isNull()) | ((F.col("cobertura_neta") >= 0) & (F.col("cobertura_neta") <= 120)))
if "desercion" in dfEstadisticas.columns:
    dfEstadisticas = dfEstadisticas.filter((F.col("desercion").isNull()) | ((F.col("desercion") >= 0) & (F.col("desercion") <= 100)))

### 3.2 Transformaciones aplicadas

**Transformación 1.** Conversión de porcentajes y cadenas numéricas a variables `double` para permitir agregaciones, correlaciones y entrenamiento de modelos.

**Transformación 2.** Agregación municipal de conectividad en instituciones oficiales y no oficiales para obtener indicadores comparables entre municipios.

**Transformación 3.** Construcción de índices derivados, entre ellos porcentaje de sedes conectadas, velocidad promedio, índice de conectividad y brechas entre sector oficial y no oficial.

**Transformación 4.** En matrícula, se consolidan los niveles educativos superiores en un total de estudiantes y se renombra la estructura para reducir errores sintácticos.

**Transformación 5.** Se construye un dataframe final integrado a nivel municipal, útil tanto para responder preguntas de negocio como para modelado supervisado y no supervisado.

In [0]:
# Transformaciones sobre Internet Oficial
if "ancho-de-banda-de-subida" in dfInternetOficiales.columns:
    dfInternetOficiales = dfInternetOficiales.withColumn("ancho_subida", F.col("ancho-de-banda-de-subida").cast("double"))
if "ancho-de-banda-descarga" in dfInternetOficiales.columns:
    dfInternetOficiales = dfInternetOficiales.withColumn("ancho_bajada", F.col("ancho-de-banda-descarga").cast("double"))
if "tiene-ancho-banda" in dfInternetOficiales.columns:
    dfInternetOficiales = dfInternetOficiales.withColumn("tiene_internet", F.col("tiene-ancho-banda").cast("double"))

# Transformaciones sobre Internet No Oficial
if "ancho-de-banda-num" in dfInternetNoOficiales.columns:
    dfInternetNoOficiales = dfInternetNoOficiales.withColumn("ancho_banda", F.col("ancho-de-banda-num").cast("double"))
if "tiene-ancho-banda" in dfInternetNoOficiales.columns:
    dfInternetNoOficiales = dfInternetNoOficiales.withColumn("tiene_internet", F.col("tiene-ancho-banda").cast("double"))
if "ancho_banda" in dfInternetNoOficiales.columns:
    dfInternetNoOficiales = dfInternetNoOficiales.withColumn("velocidad", F.col("ancho_banda"))

# Agregación municipal oficiales
aggs_of = []
if "ancho_subida" in dfInternetOficiales.columns: aggs_of.append(F.avg("ancho_subida").alias("avg_subida_of"))
if "ancho_bajada" in dfInternetOficiales.columns: aggs_of.append(F.avg("ancho_bajada").alias("avg_bajada_of"))
aggs_of.append(F.count("*").alias("total_sedes_of"))
if "tiene_internet" in dfInternetOficiales.columns: aggs_of.append(F.sum("tiene_internet").alias("sedes_con_internet_of"))

dfInternetOficialesAGG = dfInternetOficiales.groupBy("municipio_std").agg(*aggs_of)
if "avg_subida_of" in dfInternetOficialesAGG.columns and "avg_bajada_of" in dfInternetOficialesAGG.columns:
    dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn("avg_velocidad_of", (F.col("avg_subida_of") + F.col("avg_bajada_of")) / 2)
if "sedes_con_internet_of" in dfInternetOficialesAGG.columns:
    dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn("pct_sedes_internet_of", F.col("sedes_con_internet_of") / F.col("total_sedes_of"))

# Agregación municipal no oficiales
aggs_no = []
if "velocidad" in dfInternetNoOficiales.columns: aggs_no.append(F.avg("velocidad").alias("avg_velocidad_no"))
aggs_no.append(F.count("*").alias("total_sedes_no"))
if "tiene_internet" in dfInternetNoOficiales.columns: aggs_no.append(F.sum("tiene_internet").alias("sedes_con_internet_no"))

dfInternetNoOficialesAGG = dfInternetNoOficiales.groupBy("municipio_std").agg(*aggs_no)
if "sedes_con_internet_no" in dfInternetNoOficialesAGG.columns:
    dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn("pct_sedes_internet_no", F.col("sedes_con_internet_no") / F.col("total_sedes_no"))

In [0]:
# Normalización simple por ventana para indicadores de velocidad
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

if "avg_velocidad_of" in dfInternetOficialesAGG.columns:
    dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn(
        "velocidad_norm_of",
        (F.col("avg_velocidad_of") - F.min("avg_velocidad_of").over(w)) /
        (F.max("avg_velocidad_of").over(w) - F.min("avg_velocidad_of").over(w))
    )
    dfInternetOficialesAGG = dfInternetOficialesAGG.withColumn(
        "indice_conectividad_of",
        F.coalesce(F.col("velocidad_norm_of"), F.lit(0.0)) * F.coalesce(F.col("pct_sedes_internet_of"), F.lit(0.0))
    )

if "avg_velocidad_no" in dfInternetNoOficialesAGG.columns:
    dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn(
        "velocidad_norm_no",
        (F.col("avg_velocidad_no") - F.min("avg_velocidad_no").over(w)) /
        (F.max("avg_velocidad_no").over(w) - F.min("avg_velocidad_no").over(w))
    )
    dfInternetNoOficialesAGG = dfInternetNoOficialesAGG.withColumn(
        "indice_conectividad_no",
        F.coalesce(F.col("velocidad_norm_no"), F.lit(0.0)) * F.coalesce(F.col("pct_sedes_internet_no"), F.lit(0.0))
    )

In [0]:
# Transformaciones sobre Matrícula
from functools import reduce
from operator import add

if "Nombre del Municipio" in dfMatricula.columns:
    dfMatricula = dfMatricula.withColumnRenamed("Nombre del Municipio", "MUNICIPIO_ORIGINAL")

rename_map = {
    "TECNICA PROFESIONAL": "TECNICA",
    "IES CON OFERTA": "CANTIDAD_UNIVERSIDADES"
}
for k, v in rename_map.items():
    if k in dfMatricula.columns:
        dfMatricula = dfMatricula.withColumnRenamed(k, v)

niveles_superiores = [c for c in ["TECNICA", "TECNOLOGICA", "UNIVERSITARIA", "ESPECIALIZACION", "MAESTRIA", "DOCTORADO"] if c in dfMatricula.columns]
for c in niveles_superiores:
    dfMatricula = dfMatricula.withColumn(c, F.regexp_replace(F.col(c).cast("string"), "\.", "").cast("int"))

if niveles_superiores:
    dfMatricula = dfMatricula.withColumn("total_estudiantes_superior", reduce(add, [F.coalesce(F.col(c), F.lit(0)) for c in niveles_superiores]))

In [0]:
# Agregación municipal de matrícula
aggs_mat = []
if "total_estudiantes_superior" in dfMatricula.columns:
    aggs_mat.append(F.sum("total_estudiantes_superior").alias("matricula_total_superior"))
if "CANTIDAD_UNIVERSIDADES" in dfMatricula.columns:
    aggs_mat.append(F.avg(F.col("CANTIDAD_UNIVERSIDADES").cast("double")).alias("prom_universidades"))

if aggs_mat:
    dfMatriculaAGG = dfMatricula.groupBy("municipio_std").agg(*aggs_mat)
else:
    dfMatriculaAGG = dfMatricula.select("municipio_std").distinct()

# Agregación municipal de Saber 11
aggs_saber = []
for c in ["PUNT_GLOBAL", "PUNT_LECTURA_CRITICA", "PUNT_MATEMATICAS", "PUNT_C_NATURALES", "PUNT_SOCIALES_CIUDADANAS", "PUNT_INGLES"]:
    if c in dfSaber.columns:
        aggs_saber.append(F.avg(F.col(c).cast("double")).alias(f"avg_{c.lower()}"))

aggs_saber.append(F.count("*").alias("num_estudiantes_saber"))
dfSaberAGG = dfSaber.groupBy("municipio_std").agg(*aggs_saber)

# Agregación municipal de UNIDOS
if "municipio_std" in dfUNIDOS.columns:
    dfUNIDOSAGG = dfUNIDOS.groupBy("municipio_std").agg(F.count("*").alias("hogares_unidos"))
else:
    dfUNIDOSAGG = None

# Agregación municipal de Internet Fijo
if "municipio_std" in dfInternet.columns:
    cols_num_internet = []
    for c in dfInternet.columns:
        if c not in ["municipio_std", "MUNICIPIO"]:
            cols_num_internet.append(c)
    # Intento simple: contar registros por municipio como proxy de presencia de oferta fija
    dfInternetFijoAGG = dfInternet.groupBy("municipio_std").agg(F.count("*").alias("registros_internet_fijo"))
else:
    dfInternetFijoAGG = None

## 4. Integración del dataframe analítico

Una vez transformados los datos, se realiza el join a nivel municipal. Se usa como eje el municipio estandarizado, ya que este es el identificador común más estable entre los datasets disponibles. El resultado es un dataset integrado donde cada fila representa un municipio y cada columna resume una dimensión del problema: conectividad, cobertura educativa, matrícula, desempeño Saber 11 y vulnerabilidad social.

In [0]:
# Selección de variables de Estadísticas
cols_est_final = [c for c in [
    "municipio_std", "cobertura_neta", "cobertura_bruta", "desercion", "desercion_media",
    "aprobacion", "reprobacion", "repitencia", "cobertura_neta_primaria",
    "cobertura_neta_media", "tasa_matriculacion_5_16", "tamano_promedio_grupo"
] if c in dfEstadisticas.columns]

dfEstadisticasAGG = dfEstadisticas.select(*cols_est_final).dropDuplicates(["municipio_std"])

# Join progresivo
base = dfEstadisticasAGG
for extra in [dfInternetOficialesAGG, dfInternetNoOficialesAGG, dfMatriculaAGG, dfSaberAGG, dfUNIDOSAGG, dfInternetFijoAGG]:
    if extra is not None:
        base = base.join(extra, on="municipio_std", how="left")

dfFinal = base

# Variables derivadas adicionales
if "indice_conectividad_of" in dfFinal.columns and "indice_conectividad_no" in dfFinal.columns:
    dfFinal = dfFinal.withColumn("brecha_conectividad_publico_privado", F.col("indice_conectividad_of") - F.col("indice_conectividad_no"))
if "cobertura_neta_primaria" in dfFinal.columns and "cobertura_neta_media" in dfFinal.columns:
    dfFinal = dfFinal.withColumn("brecha_cobertura_primaria_media", F.col("cobertura_neta_primaria") - F.col("cobertura_neta_media"))
if "avg_punt_global" in dfFinal.columns and "desercion" in dfFinal.columns:
    dfFinal = dfFinal.withColumn("riesgo_bajo_desempeno", F.when((F.col("avg_punt_global") < 240) | (F.col("desercion") > 5), 1).otherwise(0))

print("Total municipios en dataframe final:", dfFinal.count())
print("Total columnas del dataframe final:", len(dfFinal.columns))
display(dfFinal)

## 5. Preguntas de negocio y respuesta analítica

A continuación se responden ocho preguntas de negocio alineadas con el objetivo del Ministerio de Educación. Las respuestas se construyen a partir del dataframe integrado y se acompañan de tablas que pueden usarse directamente en el documento o en la sustentación.

### Pregunta 1
**¿Cuál es la relación entre el acceso a internet en instituciones oficiales y el puntaje global promedio de Saber 11?**

In [0]:
if all(c in dfFinal.columns for c in ["pct_sedes_internet_of", "avg_punt_global"]):
    df_q1 = dfFinal.select("municipio_std", "pct_sedes_internet_of", "avg_punt_global").orderBy(F.col("pct_sedes_internet_of").desc())
    display(df_q1)
    print("Correlación:", dfFinal.stat.corr("pct_sedes_internet_of", "avg_punt_global"))

### Pregunta 2
**¿Cómo cambia la deserción cuando la conectividad oficial es baja frente a cuando es alta?**

In [0]:
if all(c in dfFinal.columns for c in ["pct_sedes_internet_of", "desercion"]):
    df_q2 = dfFinal.withColumn(
        "grupo_conectividad",
        F.when(F.col("pct_sedes_internet_of") < 0.4, "Baja")
         .when(F.col("pct_sedes_internet_of") < 0.7, "Media")
         .otherwise("Alta")
    ).groupBy("grupo_conectividad").agg(
        F.avg("desercion").alias("desercion_promedio"),
        F.count("*").alias("municipios")
    ).orderBy("grupo_conectividad")
    display(df_q2)

### Pregunta 3
**¿Qué municipios presentan mayor brecha de cobertura entre primaria y media?**

In [0]:
if "brecha_cobertura_primaria_media" in dfFinal.columns:
    df_q3 = dfFinal.select("municipio_std", "brecha_cobertura_primaria_media").orderBy(F.col("brecha_cobertura_primaria_media").desc())
    display(df_q3.limit(15))

### Pregunta 4
**¿Qué municipios combinan bajo puntaje Saber 11 con alta deserción, por lo que deberían ser priorizados?**

In [0]:
if all(c in dfFinal.columns for c in ["avg_punt_global", "desercion"]):
    df_q4 = dfFinal.select("municipio_std", "avg_punt_global", "desercion").orderBy(F.col("avg_punt_global").asc(), F.col("desercion").desc())
    display(df_q4.limit(15))

### Pregunta 5
**¿Existe relación entre matrícula en educación superior y resultados de Saber 11?**

In [0]:
if all(c in dfFinal.columns for c in ["matricula_total_superior", "avg_punt_global"]):
    df_q5 = dfFinal.select("municipio_std", "matricula_total_superior", "avg_punt_global").orderBy(F.col("matricula_total_superior").desc())
    display(df_q5)
    print("Correlación:", dfFinal.stat.corr("matricula_total_superior", "avg_punt_global"))

### Pregunta 6
**¿Cómo se relaciona la vulnerabilidad social, aproximada por los registros de UNIDOS, con el desempeño académico?**

In [0]:
if all(c in dfFinal.columns for c in ["hogares_unidos", "avg_punt_global"]):
    df_q6 = dfFinal.select("municipio_std", "hogares_unidos", "avg_punt_global").orderBy(F.col("hogares_unidos").desc())
    display(df_q6)
    print("Correlación:", dfFinal.stat.corr("hogares_unidos", "avg_punt_global"))

### Pregunta 7
**¿Qué municipios muestran mejores condiciones integrales al combinar conectividad, baja deserción y buen puntaje global?**

In [0]:
if all(c in dfFinal.columns for c in ["indice_conectividad_of", "desercion", "avg_punt_global"]):
    df_q7 = dfFinal.withColumn(
        "indice_integral",
        F.coalesce(F.col("indice_conectividad_of"), F.lit(0.0)) +
        F.coalesce(F.col("avg_punt_global")/500.0, F.lit(0.0)) -
        F.coalesce(F.col("desercion")/100.0, F.lit(0.0))
    ).select("municipio_std", "indice_integral", "indice_conectividad_of", "avg_punt_global", "desercion").orderBy(F.col("indice_integral").desc())
    display(df_q7.limit(15))

### Pregunta 8
**¿Qué municipios tienen mayor rezago simultáneo en cobertura, conectividad y resultados Saber 11?**

In [0]:
cols_needed = [c for c in ["cobertura_neta", "indice_conectividad_of", "avg_punt_global"] if c in dfFinal.columns]
if len(cols_needed) == 3:
    df_q8 = dfFinal.withColumn(
        "indice_rezago",
        (1 - F.coalesce(F.col("cobertura_neta")/100.0, F.lit(0.0))) +
        (1 - F.coalesce(F.col("indice_conectividad_of"), F.lit(0.0))) +
        (1 - F.coalesce(F.col("avg_punt_global")/500.0, F.lit(0.0)))
    ).select("municipio_std", "indice_rezago", "cobertura_neta", "indice_conectividad_of", "avg_punt_global").orderBy(F.col("indice_rezago").desc())
    display(df_q8.limit(15))

## 6. Selección de técnicas de aprendizaje de máquina

Se selecciona una técnica supervisada y una no supervisada, tal como lo exige la guía. La elección se justifica en función del objetivo del negocio y de la naturaleza de las variables construidas.

### Técnica supervisada: Random Forest Regressor
Se elige un modelo de bosque aleatorio para predecir el puntaje global promedio de Saber 11 por municipio. Esta técnica es adecuada porque tolera relaciones no lineales, interacciones entre variables y no exige supuestos estrictos de normalidad. Además, es robusta frente a valores atípicos moderados y permite trabajar bien con un conjunto mixto de indicadores educativos, sociales y de conectividad.

### Técnica no supervisada: K-Means
Se elige K-Means para segmentar municipios en perfiles territoriales. Esta técnica es útil porque permite identificar grupos homogéneos de municipios con patrones similares de conectividad, desempeño académico, cobertura y vulnerabilidad. El resultado puede convertirse en una herramienta de priorización de política pública: municipios rezagados, intermedios y destacados.

## 7. Preparación de datos para modelado

En esta etapa se eliminan variables altamente correlacionadas, se normalizan variables numéricas cuando es pertinente y se seleccionan variables según criterio de negocio. El principio rector fue mantener un conjunto compacto de predictores interpretables y útiles para explicar el desempeño territorial.

In [0]:
# Selección preliminar de variables numéricas relevantes para modelado
columnas_modelo = [c for c, t in dfFinal.dtypes if t in ["double", "int", "bigint", "float", "long"]]

# Variable objetivo para el modelo supervisado
objetivo = "avg_punt_global" if "avg_punt_global" in columnas_modelo else None

# Variables candidatas, excluyendo conteos o targets secundarios si hace falta
features_candidatas = [c for c in columnas_modelo if c != objetivo]
print("Objetivo:", objetivo)
print("Total features candidatas:", len(features_candidatas))
print(features_candidatas)

In [0]:
# Correlación respecto al target y matriz reducida para depuración de multicolinealidad
if objetivo:
    correlaciones = []
    for c in features_candidatas:
        try:
            corr = dfFinal.stat.corr(c, objetivo)
            correlaciones.append((c, corr))
        except:
            pass
    correlaciones_ordenadas = sorted(correlaciones, key=lambda x: abs(x[1]) if x[1] is not None else -1, reverse=True)
    print("Correlaciones con el target:")
    for x in correlaciones_ordenadas:
        print(x)

In [0]:
# Eliminación manual razonada de características fuertemente correlacionadas o redundantes
features_finales = [c for c in features_candidatas if c not in [
    "avg_subida_of", "avg_bajada_of",  # reemplazadas por avg_velocidad_of
    "velocidad_norm_of", "velocidad_norm_no",  # ya capturadas por índices de conectividad
    "sedes_con_internet_of", "sedes_con_internet_no",  # altamente ligadas a porcentajes
    "total_sedes_of", "total_sedes_no"  # tamaño más que desempeño
]]

# Se conservan solo columnas existentes
features_finales = [c for c in features_finales if c in dfFinal.columns and c != objetivo]
print("Features finales usadas:", features_finales)

In [0]:
# Imputación básica para modelado y preparación de dataframe final
cols_para_modelo = ([objetivo] if objetivo else []) + features_finales

# Relleno por 0 o media según lógica simple para que MLlib procese el dataset
resumen_medias = dfFinal.select([F.avg(F.col(c)).alias(c) for c in cols_para_modelo if c in dfFinal.columns]).collect()[0].asDict()
fill_dict = {k: (0.0 if v is None else float(v)) for k, v in resumen_medias.items()}

dfModel = dfFinal
for c in cols_para_modelo:
    if c in dfModel.columns:
        dfModel = dfModel.withColumn(c, F.col(c).cast("double"))
dfModel = dfModel.fillna(fill_dict)

dfModel.select([c for c in ["municipio_std", objetivo] + features_finales[:8] if c in dfModel.columns]).show(10)

### Justificación de variables utilizadas

Las variables seleccionadas representan cinco dimensiones del problema territorial: desempeño educativo previo (deserción, aprobación, repitencia), conectividad educativa (porcentaje de sedes con internet e índices de conectividad), cobertura y acceso (cobertura neta, tasa de matriculación), oportunidad educativa futura (matrícula en educación superior y presencia de instituciones) y vulnerabilidad social (UNIDOS e internet fijo como aproximaciones contextuales).

In [0]:
# Ensamble y normalización para modelos no supervisados
assembler = VectorAssembler(inputCols=features_finales, outputCol="features_raw", handleInvalid="skip")
assembled = assembler.transform(dfModel)

scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(assembled)
scaled_df = scaler_model.transform(assembled)

scaled_df.select("municipio_std", "features_raw", "features_scaled").show(5, truncate=False)

## 8. Modelo supervisado con MLlib

Se entrena un Random Forest Regressor para predecir el puntaje global promedio de Saber 11 a partir de variables territoriales. Se realizan pruebas con distintos hiperparámetros para comparar desempeño mediante RMSE y R2.

In [0]:
if objetivo:
    datos_sup = scaled_df.select("municipio_std", F.col(objetivo).alias("label"), "features_raw")
    train_df, test_df = datos_sup.randomSplit([0.8, 0.2], seed=42)

    configuraciones = [
        {"numTrees": 20, "maxDepth": 5},
        {"numTrees": 50, "maxDepth": 7},
        {"numTrees": 100, "maxDepth": 10},
    ]

    resultados_rf = []
    mejor_modelo = None
    mejor_rmse = float("inf")

    for cfg in configuraciones:
        rf = RandomForestRegressor(
            featuresCol="features_raw",
            labelCol="label",
            predictionCol="prediction",
            numTrees=cfg["numTrees"],
            maxDepth=cfg["maxDepth"],
            seed=42
        )
        modelo = rf.fit(train_df)
        pred = modelo.transform(test_df)
        eval_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
        eval_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
        rmse = eval_rmse.evaluate(pred)
        r2 = eval_r2.evaluate(pred)
        resultados_rf.append((cfg["numTrees"], cfg["maxDepth"], rmse, r2))
        if rmse < mejor_rmse:
            mejor_rmse = rmse
            mejor_modelo = modelo

    print("Resultados Random Forest (numTrees, maxDepth, rmse, r2):")
    for r in resultados_rf:
        print(r)

    pred_mejor = mejor_modelo.transform(test_df)
    display(pred_mejor.select("municipio_std", "label", "prediction"))

## 9. Modelo no supervisado con MLlib

Se aplica K-Means sobre las variables normalizadas para segmentar municipios. Se prueban varios valores de `k` y se compara el índice silhouette para seleccionar la configuración más adecuada.

In [0]:
resultados_kmeans = []
mejor_k = None
mejor_silhouette = -1
mejor_modelo_k = None

for k in [2, 3, 4, 5]:
    km = KMeans(featuresCol="features_scaled", predictionCol="cluster", k=k, seed=42)
    modelo_k = km.fit(scaled_df)
    pred_k = modelo_k.transform(scaled_df)
    evaluator = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="cluster", metricName="silhouette")
    silhouette = evaluator.evaluate(pred_k)
    resultados_kmeans.append((k, silhouette))
    if silhouette > mejor_silhouette:
        mejor_silhouette = silhouette
        mejor_k = k
        mejor_modelo_k = modelo_k

print("Resultados KMeans (k, silhouette):")
for r in resultados_kmeans:
    print(r)

segmentacion = mejor_modelo_k.transform(scaled_df)
display(segmentacion.select("municipio_std", "cluster", *[c for c in ["avg_punt_global", "desercion", "indice_conectividad_of", "cobertura_neta"] if c in segmentacion.columns]))